In [1]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.ldamulticore import LdaMulticore
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords
import pandas as pd

# Example function for text preprocessing
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    tokens = simple_preprocess(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

# Load central bank speeches data
# Replace 'your_speeches.csv' with the path to your CSV file containing speeches
speeches_df = pd.read_excel('FEDdatasentiment222.xlsx')

# Preprocess speeches
speeches_df['processed_text'] = speeches_df['text'].apply(preprocess_text)

# Create dictionary and corpus
dictionary = corpora.Dictionary(speeches_df['processed_text'])
corpus = [dictionary.doc2bow(text) for text in speeches_df['processed_text']]

# Train LDA model
lda_model = LdaMulticore(corpus=corpus,
                          id2word=dictionary,
                          num_topics=20, # Adjust based on your expectation of the number of topics
                          workers=4,
                          passes=10)

# Function to get topic distribution for a speech
def get_topic_distribution(text):
    bow = dictionary.doc2bow(preprocess_text(text))
    topic_distribution = lda_model.get_document_topics(bow)
    return topic_distribution

C:\Users\user\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Print the topics and associated words
topics = lda_model.print_topics(num_words=10)  # Adjust the number of words shown as needed

for topic in topics:
    print(topic)


(0, '0.044*"bank" + 0.021*"community" + 0.011*"banking" + 0.008*"regulatory" + 0.008*"regulation" + 0.007*"federal" + 0.007*"supervisory" + 0.006*"supervision" + 0.006*"area" + 0.006*"risk"')
(1, '0.024*"inflation" + 0.014*"rate" + 0.011*"percent" + 0.010*"price" + 0.010*"year" + 0.009*"policy" + 0.009*"labor" + 0.009*"growth" + 0.008*"economy" + 0.007*"economic"')
(2, '0.027*"risk" + 0.017*"bank" + 0.017*"basel" + 0.014*"capital" + 0.012*"ii" + 0.011*"management" + 0.008*"institution" + 0.008*"banking" + 0.006*"organization" + 0.006*"market"')
(3, '0.021*"financial" + 0.016*"country" + 0.013*"market" + 0.013*"capital" + 0.008*"risk" + 0.007*"bank" + 0.007*"growth" + 0.007*"institution" + 0.006*"economy" + 0.006*"system"')
(4, '0.010*"community" + 0.010*"market" + 0.010*"housing" + 0.008*"property" + 0.008*"neighborhood" + 0.007*"development" + 0.007*"financial" + 0.007*"data" + 0.006*"return" + 0.006*"foreclosure"')
(5, '0.037*"bank" + 0.026*"community" + 0.015*"business" + 0.015*"sma

In [3]:
# Print topics identified by LDA model
for topic_idx, topic in lda_model.print_topics():
    print(topic)
    print()
#don't need 2, 5, 9, 11, 15, 16, 18

0.044*"bank" + 0.021*"community" + 0.011*"banking" + 0.008*"regulatory" + 0.008*"regulation" + 0.007*"federal" + 0.007*"supervisory" + 0.006*"supervision" + 0.006*"area" + 0.006*"risk"

0.024*"inflation" + 0.014*"rate" + 0.011*"percent" + 0.010*"price" + 0.010*"year" + 0.009*"policy" + 0.009*"labor" + 0.009*"growth" + 0.008*"economy" + 0.007*"economic"

0.027*"risk" + 0.017*"bank" + 0.017*"basel" + 0.014*"capital" + 0.012*"ii" + 0.011*"management" + 0.008*"institution" + 0.008*"banking" + 0.006*"organization" + 0.006*"market"

0.021*"financial" + 0.016*"country" + 0.013*"market" + 0.013*"capital" + 0.008*"risk" + 0.007*"bank" + 0.007*"growth" + 0.007*"institution" + 0.006*"economy" + 0.006*"system"

0.010*"community" + 0.010*"market" + 0.010*"housing" + 0.008*"property" + 0.008*"neighborhood" + 0.007*"development" + 0.007*"financial" + 0.007*"data" + 0.006*"return" + 0.006*"foreclosure"

0.037*"bank" + 0.026*"community" + 0.015*"business" + 0.015*"small" + 0.014*"loan" + 0.011*"lending

In [4]:
#don't need 2, 5, 9, 11, 15, 16, 18
# Define topics of interest
topics_of_interest = ['economic growth', 'monetary policy', 'financial risk', 'inflation', 'global economy', 'inflation and market', 'economic policy', 'monetary policy and interest rates', 'economic growth and trade', 'monetary policy and global economy', 'economic policy and inflation', 'monetary policy and climate change', 'financial market and asset prices'] # Adjust as needed

# Filter speeches based on topics of interest
relevant_speeches = []
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    # Check if any of the topics of interest have significant probability in the topic distribution
    for topic, prob in topic_distribution:
        if lda_model.print_topic(topic) in topics_of_interest and prob > 0.2: # Adjust probability threshold as needed
            relevant_speeches.append(speech)
            break

# Now relevant_speeches contains the filtered speeches that match the topics of interest


In [5]:
print(relevant_speeches)

[]


In [6]:
# Filter speeches based on topics of interest directly from DataFrame
topics_of_interest = [2, 5, 9] # Adjust as needed

relevant_speeches_df = pd.DataFrame(columns=speeches_df.columns)  # Create an empty DataFrame with the same columns as speeches_df

for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    # Check if any of the topics of interest have significant probability in the topic distribution
    for topic, prob in topic_distribution:
        if lda_model.print_topic(topic) in topics_of_interest and prob > 0.2: # Adjust probability threshold as needed
            relevant_speeches_df = relevant_speeches_df.append(speech, ignore_index=True)
            break

# Now relevant_speeches_df contains only the relevant speeches


In [9]:
relevant_speeches_df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Link URL,Author,date,Title,text,sentiment vader,vader_sentiment,positive_score,negative_score,uncertainty_score,Polarity_Score,processed_text


In [10]:
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    print(f"Speech {idx} - Topic Distribution: {topic_distribution}")


Speech 0 - Topic Distribution: [(1, 0.7445404), (4, 0.018798122), (11, 0.14834422), (19, 0.08775077)]
Speech 1 - Topic Distribution: [(2, 0.8607483), (16, 0.13825937)]
Speech 2 - Topic Distribution: [(5, 0.21194439), (6, 0.41135848), (12, 0.31939813), (17, 0.05440221)]
Speech 3 - Topic Distribution: [(2, 0.044782463), (3, 0.16577283), (8, 0.06434845), (9, 0.5779154), (12, 0.11295422), (17, 0.03095386)]
Speech 4 - Topic Distribution: [(8, 0.012105646), (19, 0.98731583)]
Speech 5 - Topic Distribution: [(1, 0.7396689), (8, 0.24934733), (17, 0.010579843)]
Speech 6 - Topic Distribution: [(0, 0.4189778), (1, 0.041250106), (2, 0.071567014), (5, 0.35912457), (9, 0.03861317), (17, 0.062864505)]
Speech 7 - Topic Distribution: [(0, 0.027069058), (2, 0.16867462), (3, 0.4714422), (9, 0.2976105), (12, 0.034387555)]
Speech 8 - Topic Distribution: [(3, 0.03961365), (5, 0.03144069), (9, 0.32375538), (11, 0.033950314), (12, 0.2526618), (13, 0.27334794), (16, 0.039684553)]
Speech 9 - Topic Distribution: 

In [30]:
import pandas as pd

# Define the indices of the topics you are interested in
topics_of_interest = [0, 1, 4]  # Adjust based on your topics of interest

# Create an empty DataFrame to store relevant speeches
relevant_speeches_df = pd.DataFrame(columns=speeches_df.columns)

# Iterate through each speech and filter based on topics of interest
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    # Check if any of the topics of interest have significant probability in the topic distribution
    relevant_topics = [topic for topic, prob in topic_distribution if topic in topics_of_interest and prob > 0.2]
    if relevant_topics:
        relevant_speeches_df = relevant_speeches_df.append(speech, ignore_index=True)

# Print or do something else with the relevant speeches DataFrame
print(relevant_speeches_df)


AttributeError: 'DataFrame' object has no attribute 'append'

In [29]:
speeches_df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Link URL,Author,date,Title,text,sentiment vader,vader_sentiment,positive_score,negative_score,uncertainty_score,Polarity_Score,processed_text
0,0,0,1053,https://www.federalreserve.gov/newsevents/spee...,Governor Susan Schmidt Bies,2006-01-18,Productivity and Economic Outlook,appreciate opportunity speak today productivit...,positive,0.225877,60402393,118486857,81571443,0.200000,"[appreciate, opportunity, speak, today, produc..."
1,1,1,1052,https://www.federalreserve.gov/newsevents/spee...,Governor Susan Schmidt Bies,2006-02-02,The Continuous Challenges of Risk Management,thank invitation speak today remark focus cont...,positive,0.196206,60402393,118486857,81571443,-0.157895,"[thank, invitation, speak, today, remark, focu..."
2,2,2,1051,https://www.federalreserve.gov/newsevents/spee...,Chairman Ben S. Bernanke,2006-02-06,Remarks at ceremonial swearing-in by President...,good morning would like begin thanking preside...,positive,0.742904,60402393,118486857,81571443,0.793103,"[good, morning, would, like, begin, thanking, ..."
3,3,3,1050,https://www.federalreserve.gov/newsevents/spee...,"Vice Chairman Roger W. Ferguson, Jr.",2006-02-23,"Globalization, Insurers, and Regulators: Share...",honored deliver keynote address important inte...,positive,0.423985,60402393,118486857,81571443,0.156463,"[honored, deliver, keynote, address, important..."
4,4,4,1048,https://www.federalreserve.gov/newsevents/spee...,"Vice Chairman Roger W. Ferguson, Jr.",2006-02-24,The Importance of Education,pleased opportunity part applied physic labora...,positive,0.414621,60402393,118486857,81571443,0.301205,"[pleased, opportunity, part, applied, physic, ..."


In [34]:
#don't need 2, 5, 9, 11, 15, 16, 18
import pandas as pd

# Define the indices of the topics you are interested in
topics_of_interest = [0, 1, 3, 4, 6, 7, 8, 10, 12, 13, 14, 17, 19]  # Adjust based on your topics of interest

# Create an empty list to store relevant speeches
relevant_speeches = []

# Iterate through each speech and filter based on topics of interest
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    # Check if any of the topics of interest have significant probability in the topic distribution
    relevant_topics = [topic for topic, prob in topic_distribution if topic in topics_of_interest and prob > 0.2]
    if relevant_topics:
        relevant_speeches.append(speech)

# Convert the list of relevant speeches into a DataFrame
relevant_speeches_df = pd.DataFrame(relevant_speeches)

# Print or do something else with the relevant speeches DataFrame
print(relevant_speeches_df)


     Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0  \
0               0             0        1053   
2               2             2        1051   
3               3             3        1050   
4               4             4        1048   
5               5             5        1047   
..            ...           ...         ...   
890           890           890          18   
891           891           891          14   
893           893           893          11   
896           896           896           7   
900           900           900           0   

                                              Link URL  \
0    https://www.federalreserve.gov/newsevents/spee...   
2    https://www.federalreserve.gov/newsevents/spee...   
3    https://www.federalreserve.gov/newsevents/spee...   
4    https://www.federalreserve.gov/newsevents/spee...   
5    https://www.federalreserve.gov/newsevents/spee...   
..                                                 ...   
890  https://www.federalreser

In [32]:
speeches_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Link URL,Author,date,Title,text,sentiment vader,vader_sentiment,positive_score,negative_score,uncertainty_score,Polarity_Score,processed_text
896,896,896,7,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2023-11-28,Something Appears to Be Giving,last month gave speech entitled somethings got...,positive,0.270940,60402393,118486857,81571443,-0.220779,"[last, month, gave, speech, entitled, somethin..."
897,897,897,4,https://www.federalreserve.gov/newsevents/spee...,Vice Chair for Supervision Michael S. Barr,2023-12-01,The Importance of Effective Liquidity Risk Man...,thank opportunity speak im delighted celebrate...,positive,0.079550,60402393,118486857,81571443,-0.526786,"[thank, opportunity, speak, im, delighted, cel..."
898,898,898,2,https://www.federalreserve.gov/newsevents/spee...,Governor Michelle W. Bowman,2023-12-05,Building a More Inclusive Financial System thr...,would like thank aspen institute inviting join...,positive,0.385244,60402393,118486857,81571443,0.482759,"[would, like, thank, aspen, institute, invitin..."
899,899,899,1,https://www.federalreserve.gov/newsevents/spee...,Governor Michelle W. Bowman,2024-01-08,New Year’s Resolutions for Bank Regulatory Pol...,pleasure join afternoon south carolina communi...,positive,0.264607,60402393,118486857,81571443,-0.313953,"[pleasure, join, afternoon, south, carolina, c..."
900,900,900,0,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2024-01-16,Almost as Good as It Gets…But Will It Last?,thank david wessel thank brookings opportunity...,positive,0.083225,60402393,118486857,81571443,-0.321739,"[thank, david, wessel, thank, brookings, oppor..."


In [33]:
relevant_speeches_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Link URL,Author,date,Title,text,sentiment vader,vader_sentiment,positive_score,negative_score,uncertainty_score,Polarity_Score,processed_text
874,874,874,41,https://www.federalreserve.gov/newsevents/spee...,Governor Michelle W. Bowman,2023-09-26,Welcoming Remarks,welcome thank joining u federal reserve commun...,positive,0.115287,60402393,118486857,81571443,0.069767,"[welcome, thank, joining, federal, reserve, co..."
881,881,881,31,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2023-10-10,Monetary Policy Analysis and the Development o...,pleasure honor legacy bennett mccallum discuss...,positive,0.220939,60402393,118486857,81571443,0.044248,"[pleasure, honor, legacy, bennett, mccallum, d..."
884,884,884,28,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2023-10-18,Something’s Got to Give,thank professor scobie thank european economic...,positive,0.352477,60402393,118486857,81571443,-0.149425,"[thank, professor, scobie, thank, european, ec..."
896,896,896,7,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2023-11-28,Something Appears to Be Giving,last month gave speech entitled somethings got...,positive,0.270940,60402393,118486857,81571443,-0.220779,"[last, month, gave, speech, entitled, somethin..."
900,900,900,0,https://www.federalreserve.gov/newsevents/spee...,Governor Christopher J. Waller,2024-01-16,Almost as Good as It Gets…But Will It Last?,thank david wessel thank brookings opportunity...,positive,0.083225,60402393,118486857,81571443,-0.321739,"[thank, david, wessel, thank, brookings, oppor..."


In [36]:
relevant_speeches_df.to_excel('FEDdatasentimentfinal44.xlsx', index=False)

In [2]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.ldamulticore import LdaMulticore
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords
import pandas as pd

In [5]:
import pandas as pd
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

# Example function for text preprocessing
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    
    # Convert non-string values to string
    if not isinstance(text, str):
        text = str(text)
        
    tokens = simple_preprocess(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

# Load central bank speeches data
# Replace 'your_speeches.csv' with the path to your CSV file containing speeches
speeches_df = pd.read_excel('ECBdatasentiment224.xlsx')

# Preprocess speeches
speeches_df['processed_text'] = speeches_df['text'].apply(preprocess_text)

# Create dictionary and corpus
dictionary = corpora.Dictionary(speeches_df['processed_text'])
corpus = [dictionary.doc2bow(text) for text in speeches_df['processed_text']]

# Train LDA model
lda_model = LdaMulticore(corpus=corpus,
                          id2word=dictionary,
                          num_topics=10, # Adjust based on your expectation of the number of topics
                          workers=4,
                          passes=10)

# Function to get topic distribution for a speech
def get_topic_distribution(text):
    bow = dictionary.doc2bow(preprocess_text(text))
    topic_distribution = lda_model.get_document_topics(bow)
    return topic_distribution


In [3]:
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    print(f"Speech {idx} - Topic Distribution: {topic_distribution}")

Speech 0 - Topic Distribution: [(0, 0.096061796), (3, 0.90342)]
Speech 1 - Topic Distribution: [(1, 0.17246792), (15, 0.8163452)]
Speech 2 - Topic Distribution: [(0, 0.12883228), (1, 0.17031251), (2, 0.012405928), (3, 0.4210172), (7, 0.074726395), (12, 0.18291551)]
Speech 3 - Topic Distribution: [(1, 0.35859624), (2, 0.5473718), (12, 0.09304058)]
Speech 4 - Topic Distribution: [(1, 0.6171017), (3, 0.09790273), (7, 0.16210887), (12, 0.12085614)]
Speech 5 - Topic Distribution: [(1, 0.029553413), (2, 0.3672291), (9, 0.5892772), (12, 0.012769013)]
Speech 6 - Topic Distribution: [(1, 0.10283034), (2, 0.016563376), (3, 0.033634238), (6, 0.2851981), (15, 0.5274335), (16, 0.029394027)]
Speech 7 - Topic Distribution: [(0, 0.13107143), (1, 0.02321957), (2, 0.12432649), (3, 0.35801253), (6, 0.095100075), (7, 0.22072478), (12, 0.046561845)]
Speech 8 - Topic Distribution: [(1, 0.49323723), (3, 0.3720982), (6, 0.13400923)]
Speech 9 - Topic Distribution: [(2, 0.52395874), (6, 0.10336803), (12, 0.3718

In [6]:
# Print topics identified by LDA model
for topic_idx, topic in lda_model.print_topics():
    print(topic)
    print()

0.030*"financial" + 0.014*"risk" + 0.013*"bank" + 0.010*"banking" + 0.009*"market" + 0.008*"policy" + 0.008*"stability" + 0.008*"system" + 0.007*"macroprudential" + 0.007*"crisis"

0.015*"payment" + 0.013*"bank" + 0.009*"market" + 0.009*"european" + 0.009*"financial" + 0.007*"euro" + 0.007*"also" + 0.007*"ha" + 0.006*"would" + 0.006*"service"

0.022*"bank" + 0.021*"market" + 0.016*"central" + 0.013*"financial" + 0.011*"policy" + 0.011*"liquidity" + 0.010*"monetary" + 0.008*"money" + 0.008*"risk" + 0.008*"rate"

0.015*"financial" + 0.013*"policy" + 0.013*"euro" + 0.013*"area" + 0.011*"monetary" + 0.011*"bank" + 0.009*"market" + 0.009*"ha" + 0.009*"crisis" + 0.008*"price"

0.039*"die" + 0.035*"der" + 0.027*"und" + 0.012*"zu" + 0.011*"den" + 0.010*"ist" + 0.009*"von" + 0.009*"fã¼r" + 0.009*"da" + 0.009*"im"

0.063*"de" + 0.038*"la" + 0.019*"le" + 0.016*"en" + 0.011*"que" + 0.011*"et" + 0.009*"el" + 0.008*"los" + 0.007*"euro" + 0.006*"un"

0.019*"euro" + 0.015*"area" + 0.012*"country" + 0.

In [7]:
import pandas as pd

# Define the indices of the topics you are interested in
topics_of_interest = [0, 1, 3, 6, 7, 8]  # Adjust based on your topics of interest

# Create an empty list to store relevant speeches
relevant_speeches = []

# Iterate through each speech and filter based on topics of interest
for idx, speech in speeches_df.iterrows():
    topic_distribution = get_topic_distribution(speech['text'])
    # Check if any of the topics of interest have significant probability in the topic distribution
    relevant_topics = [topic for topic, prob in topic_distribution if topic in topics_of_interest and prob > 0.2]
    if relevant_topics:
        relevant_speeches.append(speech)

# Convert the list of relevant speeches into a DataFrame
relevant_speeches_df = pd.DataFrame(relevant_speeches)

# Print or do something else with the relevant speeches DataFrame
print(relevant_speeches_df)

      Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0       date  \
0                0          1971        2023 2006-01-13   
2                2          1969        2021 2006-01-20   
3                3          1968        2020 2006-02-02   
4                4          1967        2019 2006-02-06   
5                5          1966        2018 2006-02-10   
...            ...           ...         ...        ...   
1488          1488             5           5 2023-11-21   
1490          1490             3           3 2023-11-27   
1491          1491             2           2 2023-11-30   
1492          1492             1           1 2023-12-04   
1493          1493             0           0 2023-12-07   

                            speakers  \
0                Jean-Claude Trichet   
2                Jean-Claude Trichet   
3     JosÃ© Manuel GonzÃ¡lez-PÃ¡ramo   
4                Jean-Claude Trichet   
5                    Lucas Papademos   
...                              ...   
1488       

In [8]:
relevant_speeches_df.to_excel('ECBdatasentimentfinal4ln.xlsx', index=False)